In [137]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [138]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
from src.config import get_config
from src.models import build_model

In [139]:
CHEXPERT_LABELS = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

In [140]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Change to config you want to use
rel_path_to_config = "configs/simmim_finetune__vit_base__img224__100ep.yaml"

class Args:
    def __init__(self):
        self.cfg = "../" + rel_path_to_config
        self.opts = None
        self.batch_size = None
        self.data_path = None
        self.resume = None
        self.accumulation_steps = None
        self.use_checkpoint = False
        self.amp_opt_level = "O0"
        self.output = "output"
        self.tag = None
        self.local_rank = 0
        self.pretrained = None
        self.eval = False
        self.throughput = False
args = Args()
config = get_config(args)

config.defrost()
config.MODEL.NUM_CLASSES = 14
config.freeze()

model = build_model(config, is_pretrain=False)
ckpt_path = "../simmim_checkpoints/simmim_finetune/ckpt_epoch_99.pth" # change to model weights you want to use 
ckpt = torch.load(ckpt_path, map_location="cpu")

dictt = ckpt["model"]
new_dict = {}
for k, v in dictt.items():
    new_k = k.replace("module.","")
    new_dict[new_k] = v

model.load_state_dict(new_dict, strict=False)
model.eval()
print("Model loaded")


Configuration file loaded at ../configs ../configs/simmim_finetune__vit_base__img224__100ep.yaml
Model loaded


In [141]:
CHEXPERT_MEAN = [0.533430]
CHEXPERT_STD = [0.283167]

transform = transforms.Compose([
    transforms.Resize(config.DATA.IMG_SIZE),
    transforms.CenterCrop(config.DATA.IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = CHEXPERT_MEAN,
        std = CHEXPERT_STD
    ),
])

In [ ]:
chexpert_normal_path = "./demo_xrays/test_chexpert/no_finding/1.jpg"
chexpert_lung_opacity_path = "./demo_xrays/test_chexpert/lung_opacity/1.jpg"
pneumonia_path = "./demo_xrays/pneumonia/pneumonia_2.jpeg"    

image_path = pneumonia_path

# image_path = "./demo_xrays/normal/normal_2.jpeg" # change to image you want to run the inference on
image = Image.open(image_path).convert("L")

plt.imshow(image, cmap="gray")
plt.title("Input X-ray")
plt.axis("off")

In [ ]:
input_tensor = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(input_tensor)
    probs = torch.sigmoid(outputs).cpu().numpy()[0]

# threshold = 0.5

k = 6
top_k = sorted(
    list(zip(CHEXPERT_LABELS, probs)),
    key=lambda x: x[1],
    reverse=True
)

print("Top Predictions:\n")
for label, prob in top_k[:k]:
    if label != "Support Devices": # ignoring because not pathology related
        print(f"{label:20s} {prob:.3f}")


In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(image, cmap="gray")
plt.axis("off")


text = "\n".join([
    f"{label}: {prob:.2f}" for label, prob in top_k[:k] if label != "Support Devices"
])

plt.title("Top Model Predictions", fontsize=14)
plt.text(
    1.05, 0.5, text,
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment="center"
)

plt.show()